# What CRG missed on our 5 projects, and why
Every row CRG missed (real per the compiler, not in CRG's graph), sorted into one reason by a syntax test on its source line.
Sources: functions = `results/treesitter-module/2026-11-18_1555/who_found_what.csv` (same CRG graphs and counts as the fresh run); classes = `class_lab.py` (as `class-crg-<project>.ipynb`); edges = newest `results/<project>/checks/edges_*/*_missed.csv`.
Each reason is checked on the source line of the row; rows no test explains are 'other' and listed with examples.

In [1]:
import pandas as pd, glob, re, class_lab as L
R = L.ROOT / 'results'; APPS = L.ROOT / 'apps'
P = {'guava': 'java', 'spring-petclinic': 'java', 'crm-prototype': 'javascript', 'crm-prototype-client': 'javascript', 'voice-assessment-agent': 'python'}
RUN = {'guava': 'iter-003_2026-11-17_2247', 'spring-petclinic': 'iter-002_2026-11-17_2008', 'crm-prototype': 'iter-003_2026-11-18_0743',
       'crm-prototype-client': 'iter-006_2026-11-18_0744', 'voice-assessment-agent': 'iter-005_2026-11-17_2251'}
_lines = {}
def line(p, f, n):
    k = (p, f)
    if k not in _lines:
        _lines[k] = (APPS / p / f).read_text(errors='replace').splitlines()
    L_ = _lines[k]; n = int(n)
    return L_[n - 1].strip() if 0 < n <= len(L_) else ''
rows = []
def add(p, kind, reason, r_file, r_line, src):
    rows.append(dict(project=p, kind=kind, reason=reason, file=r_file, line=int(r_line), source=src[:160]))
def decl(p, f, n):
    n = int(n); s = line(p, f, n)
    while s.startswith('@'):
        n += 1; s = line(p, f, n)
    return s
PROP_ASSIGN = re.compile(r'^[\w$.]+\.[\w$]+\s*=\s*(async\s+)?(function\b|\(|[\w$]+\s*=>)')

## Functions

In [2]:
w = pd.read_csv(R / 'treesitter-module/2026-11-18_1555/who_found_what.csv')
miss = w[w.who.isin(['module only', 'neither'])]
for r in miss.itertuples():
    s = line(r.app, r.file, r.line); same = ((w.app == r.app) & (w.file == r.file) & (w.function == r.function)).sum()
    if re.match(r'^[\w$]+\s*:\s*(async\s+)?(function\b|\(|[\w$]+\s*=>)', s): why = 'function written as an object property (key: function / key: () =>)'
    elif PROP_ASSIGN.match(s): why = 'function assigned to an object property (obj.onload = () => ...)'
    elif 'useCallback(' in s or 'useMemo(' in s: why = 'function wrapped in useCallback / useMemo'
    elif re.search(r'\)\s*default\b', s) or (P[r.app] == 'java' and s.endswith(';') and '(' in s): why = 'annotation member (@interface X { String value(); })'
    elif s.startswith('@'): why = 'method with an annotation on the line above (line taken from the annotation)'
    elif same > 1: why = 'overloaded method (same name more than once in the file)'
    else: why = 'other'
    add(r.app, 'Functions', why, r.file, r.line, s)

## Classes

In [3]:
for p, lang in P.items():
    repo = APPS / p; files = L.repo_files(repo, lang)
    ref = L.reference(repo, lang, files); ref = ref[ref.error.isna()]; named = ref[ref.kind != 'anonymous'].reset_index(drop=True)
    crg = L.crg_classes(R / p / 'runs' / RUN[p] / 'crg/graph.db', repo, lang, files)
    _, missed, _ = L.score(named, crg)
    for r in missed.itertuples():
        s = line(p, r.file, r.line)
        why = ('annotation type (@interface)' if r.kind == 'annotation' or '@interface' in s
               else 'class defined inside a function' if s.startswith(('class ', '    class')) and lang == 'python' and r.kind != 'class' else 'other')
        if why == 'other' and lang == 'python':
            why = 'class nested inside another class or function' if (APPS / p / r.file).read_text(errors='replace').splitlines()[r.line - 1].startswith((' ', '\t')) else 'other'
        if why == 'other' and s.startswith('@'): why = 'class with a decorator (@dataclass): the line points at the decorator'
        add(p, 'Classes', why, r.file, r.line, s)

## Edges

In [4]:
def why_edge(p, k, r, s, m):
    how = getattr(r, 'how', '') if isinstance(getattr(r, 'how', ''), str) else ''
    if k == 'CALLS':
        if how == 'new': return 'constructor call (new X(...))'
        if how.startswith('jsx'): return 'JSX element (<div>, <Component>) counted as a call'
        if r.name in ('super', 'this'): return 'super(...) / this(...) call'
        if s.startswith('.'): return 'chained call on a continuation line (.foo(...) at the start of the line)'
        if len(re.findall(r'\b' + re.escape(str(r.name)) + r'\s*\(', s)) > 1: return 'same call more than once on one line (CRG keeps one)'
        if re.search(r'\b' + re.escape(str(r.name)) + r'\s*\(', s) and re.search(r'\(.*\b' + re.escape(str(r.name)) + r'\s*\(', s): return 'call nested inside another call\'s arguments'
        return 'other'
    if k == 'CONTAINS':
        if getattr(r, 'kind', '') == 'class' and not isinstance(r.child, str): return 'anonymous class (new X() {...}) inside a class'
        if getattr(r, 'kind', '') == 'class': return 'nested class inside a class'
        if re.match(r'^[\w$]+\s*:\s*(async\s+)?(function\b|\(|[\w$]+\s*=>)', s): return 'function written as an object property (key: function / key: () =>)'
        if PROP_ASSIGN.match(s): return 'function assigned to an object property (obj.onload = () => ...)'
        if 'useCallback(' in s or 'useMemo(' in s: return 'function wrapped in useCallback / useMemo'
        if re.search(r'\)\s*default\b', s) or (getattr(r, 'kind', '') == 'method' and s.endswith(';')): return 'annotation member (@interface X { String value(); })'
        if getattr(r, 'kind', '') == 'method' and ((m.child == r.child) & (m.file == r.file)).sum() >= 1: return 'overloaded method (same name more than once in the class)'
        return 'other'
    if k == 'INHERITS':
        if P[p] == 'javascript': return 'class X extends Y (CRG stores no inheritance for JavaScript here)'
        s = decl(p, r.file, r.line)
        if re.search(r'\binterface\b', s): return 'interface extending interfaces (interface A extends B, C)'
        if len(re.findall(r'\bextends\b|\bimplements\b', s)) and ',' in s.split('{')[0]: return 'class with more than one parent (extends A implements B, C)'
        if re.search(r'\bextends\b', s): return 'class extends a generic or nested type (extends Outer.Inner / X<T>)'
        return 'other'
    if k == 'IMPORTS_FROM':
        if s.startswith('export'): return 're-export (export ... from \'./x\')'
        if how == 'require' and not s.startswith(('const', 'let', 'var', 'import')): return 'require() inside a function, not at the top of the file'
        if how == 'require': return 'require() inside a function, not at the top of the file'
        if '__future__' in s: return 'from __future__ import ...'
        return 'other'
names = {'CONTAINS': 'Contains', 'INHERITS': 'Inherits', 'IMPORTS_FROM': 'Imports', 'CALLS': 'Calls'}
for p in P:
    d = sorted(glob.glob(str(R / p / 'checks/edges_*')))[-1]
    for k in names:
        m = pd.read_csv(f'{d}/{k}_missed.csv')
        for r in m.itertuples():
            s = line(p, r.file, r.line); add(p, names[k], why_edge(p, k, r, s, m), r.file, r.line, s)

In [5]:
t = pd.DataFrame(rows)
summary = (t.groupby(['project', 'kind', 'reason']).agg(missed=('line', 'size'), example=('source', 'first'),
           example_at=('file', 'first')).reset_index().sort_values(['project', 'kind', 'missed'], ascending=[True, True, False]))
t.to_csv(R / 'five-projects-crg-missed-rows.csv', index=False); summary.to_csv(R / 'five-projects-crg-missed-reasons.csv', index=False)
pd.set_option('display.max_colwidth', 90); pd.set_option('display.width', 250)
summary[['project', 'kind', 'reason', 'missed', 'example']]

,project,kind,reason,missed,example
1,crm-prototype,Calls,constructor call (new X(...)),169,const s3 = new S3Client(
0,crm-prototype,Calls,chained call on a continuation line (.foo(...) at the start of the line),77,".replace(/{{email}}/g, to)"
2,crm-prototype,Contains,function written as an object property (key: function / key: () =>),21,validator: function (value) {
3,crm-prototype,Functions,function written as an object property (key: function / key: () =>),21,validator: function (value) {
4,crm-prototype,Imports,"require() inside a function, not at the top of the file",3,const ROLES = require('../constants/role.constant')
5,crm-prototype,Inherits,class X extends Y (CRG stores no inheritance for JavaScript here),6,class BadRequestError extends Error {
6,crm-prototype-client,Calls,"JSX element (<div>, <Component>) counted as a call",979,"<div className=""px-4 sm:px-6 py-4 sm:py-6 space-y-4 sm:space-y-6 overflow-y-auto max-h..."
8,crm-prototype-client,Calls,constructor call (new X(...)),87,const queryClient = new QueryClient()
9,crm-prototype-client,Calls,same call more than once on one line (CRG keeps one),32,"{isInviting ? t(""inviting"") : t(""share"")}"
7,crm-prototype-client,Calls,chained call on a continuation line (.foo(...) at the start of the line),1,.use(initReactI18next)


In [6]:
# 'other' rows: examples, so nothing is hidden
t[t.reason == 'other'].groupby(['project', 'kind']).head(4)[['project', 'kind', 'file', 'line', 'source']]

,project,kind,file,line,source
4934,guava,Inherits,android/guava-tests/test/com/google/common/io/CloserTest.java,324,private static class TestSuppressor implements Closer.Suppressor {
4960,guava,Inherits,android/guava-tests/test/com/google/common/reflect/TypeTokenTest.java,1338,class SingleGenericExtendsBase<T> implements GenericClass.Base {}
5003,guava,Inherits,android/guava/src/com/google/common/hash/BloomFilterStrategies.java,41,@J2ktIncompatible
5017,guava,Inherits,android/guava/src/com/google/common/util/concurrent/ListeningExecutorService.java,36,@DoNotMock(
